In [ ]:
import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from mamba_ssm import Mamba # Załóżmy, że mamy zainstalowaną bibliotekę Mamby

class GraphMamba(nn.Module):
    def __init__(self, in_features, gnn_hidden_features, mamba_hidden_features, num_gnn_layers):
        super().__init__()

        self.in_features = in_features
        self.gnn_hidden_features = gnn_hidden_features

        # --- Blok GNN (Lokalny detektyw) ---
        self.gnn_layers = nn.ModuleList()
        # Warstwa wejściowa
        self.gnn_layers.append(GCNConv(in_features, gnn_hidden_features))
        # Ukryte warstwy GNN
        for _ in range(num_gnn_layers - 1):
            self.gnn_layers.append(GCNConv(gnn_hidden_features, gnn_hidden_features))

        # Sprawdzenie, czy wymiary się zgadzają dla Mamby
        assert gnn_hidden_features == mamba_hidden_features, "GNN output and Mamba input dims must match for residual connection"

        # --- Blok Mamby (Globalny analityk) ---
        self.mamba = Mamba(
            d_model=mamba_hidden_features, # Wymiar wejściowy/ukryty Mamby
            d_state=16,  # Standardowy hiperparametr Mamby (stan ukryty)
            d_conv=4,    # Standardowy hiperparametr Mamby (konwolucja)
            expand=2,    # Standardowy hiperparametr Mamby (ekspansja)
        )

        # Warstwa normalizacyjna dla stabilności
        self.norm = nn.LayerNorm(gnn_hidden_features)

    def forward(self, x, edge_index):
        # x: [num_nodes, in_features]
        # edge_index: [2, num_edges]

        # 1. Krok GNN: Przetwarzanie lokalne
        h_gnn = x
        for layer in self.gnn_layers:
            h_gnn = layer(h_gnn, edge_index).relu()

        # 2. Przygotowanie sekwencji dla Mamby
        # Mamba oczekuje wejścia w formacie [batch_size, sequence_length, features]
        # Traktujemy cały graf jako jedną sekwencję w batchu
        seq_in = h_gnn.unsqueeze(0) # -> [1, num_nodes, gnn_hidden_features]

        # 3. Krok Mamby: Przetwarzanie globalne
        seq_out = self.mamba(seq_in) # -> [1, num_nodes, mamba_hidden_features]

        # Usuwamy wymiar batcha
        h_mamba = seq_out.squeeze(0) # -> [num_nodes, mamba_hidden_features]

        # 4. Integracja: Połączenie rezydualne
        # Dodajemy globalny kontekst z Mamby do lokalnych cech z GNN
        h_final = self.norm(h_gnn + h_mamba)

        # h_final to twoje ostateczne, wzbogacone wektory wezłów.
        # Teraz możesz je podać do głowic decyzyjnych (np. polityki i wartości w RL).
        return h_final